## Experiments on feature engineering 

This notebook is dedicated to feature engineering experiments conducted on the cleaned dataset. First, we load the clean data.

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path
import sys

In [2]:
# Define Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / 'src'))
from ingestion import load_data

data_path = PROJECT_ROOT / 'data' / 'processed' / 'games_cleaned.json'

# Load data
df = load_data(data_path, False)
display(df.head())
print(df.info())

Loading data from /home/axel/grec/data/processed/games_cleaned.json...
Successfully loaded 22620 records.


,app_id,name,genres,tags,positive,negative,short_description
0,750920,Shadow of the Tomb Raider: Definitive Edition,"[Action, Adventure]","{'Adventure': 915, 'Action': 709, 'Female Prot...",66684,12759,As Lara Croft races to save the world from a M...
1,311210,Call of Duty®: Black Ops III,"[Action, Adventure]","{'Multiplayer': 3768, 'Zombies': 2571, 'FPS': ...",181306,33720,Call of Duty®: Black Ops III Zombies Chronicle...
2,2232200,First Snow,"[Casual, Indie]","{'Visual Novel': 127, 'Dating Sim': 120, 'Roma...",99,11,"첫눈이 내리던 날, 나는 그녀에게 첫눈에 반해 첫사랑에 빠졌다. 첫눈이 내리는 나날..."
3,1259430,First Snow,"[Casual, Indie, Free To Play]","{'LGBTQ+': 212, 'Free to Play': 211, 'Casual':...",613,36,Having moved away from her friends and family ...
4,8800,Civilization IV: Beyond the Sword,[Strategy],"{'Strategy': 182, 'Turn-Based Strategy': 130, ...",4423,169,Sid Meier's Civilization IV®: Beyond the Sword...


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22620 entries, 0 to 22619
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   app_id             22620 non-null  int64 
 1   name               22620 non-null  object
 2   genres             22620 non-null  object
 3   tags               22620 non-null  object
 4   positive           22620 non-null  int64 
 5   negative           22620 non-null  int64 
 6   short_description  22620 non-null  object
dtypes: int64(3), object(4)
memory usage: 1.2+ MB
None


We need to define what makes games similar. We can consider two games to be similar if they share many of the same tags and genres. Additionally, if their descriptions are similar, they can also be considered similar. Therefore, to quantify this similarity, we will construct a composite feature vector for each game. This unified vector will be formed by concatenating the tags vector, the multi-hot encoded genres vector, and the embedding of the description.

In [3]:
# Vector for tags

# Extract tag vocabulary
all_tags = set()
for tags_dict in df['tags']:
    all_tags.update(tags_dict.keys())

vocab_tags = sorted(all_tags)  # Sort for consistency

print(f"Vocabulary of {len(vocab_tags)} tags")

# Vector per game
def create_tag_vector(tags_dict, vocab):
    """
    Creates a feature vector for a game based on the vocabulary.
    feature_tag[i] = log(1 + tag_votes) if the game has the tag
    feature_tag[i] = 0 if it doesn't have it
    """
    vector = np.zeros(len(vocab))
    
    for i, tag in enumerate(vocab):
        if tag in tags_dict:
            vector[i] = np.log(1 + tags_dict[tag])

    return vector

# Create vectors for all games
tag_vectors = np.array([create_tag_vector(tags, vocab_tags) for tags in df['tags']])

print(f"Tag matrix shape: {tag_vectors.shape}")

non_zero_indices = np.nonzero(tag_vectors[0])[0]
print(f"First game - ALL non-zero positions ({len(non_zero_indices)} tags):")
for idx in non_zero_indices:
    print(f"  {vocab_tags[idx]}: {tag_vectors[0][idx]:.4f}")

# Normalize (L2 normalization), compare tag composition instead of tag quantity
from sklearn.preprocessing import normalize

tag_vectors_norm = normalize(tag_vectors, norm='l2', axis=1)

print(f"First game after normalization - non-zero positions):")
for idx in non_zero_indices:
    print(f"  {vocab_tags[idx]}: {tag_vectors_norm[0][idx]:.4f}")

Vocabulary of 448 tags
Tag matrix shape: (22620, 448)
First game - ALL non-zero positions (20 tags):
  Action: 6.5653
  Action-Adventure: 6.0259
  Adventure: 6.8200
  Atmospheric: 5.5094
  Dark: 4.8903
  Exploration: 5.9454
  Female Protagonist: 6.3801
  Gore: 5.2095
  Great Soundtrack: 5.0039
  Heist: 4.3307
  Multiplayer: 4.9698
  Open World: 6.2989
  Puzzle: 6.1003
  Shooter: 5.4467
  Singleplayer: 6.3190
  Stealth: 6.0403
  Story Rich: 6.2422
  Survival: 5.8171
  Third Person: 6.2634
  Violent: 5.5452
First game after normalization - non-zero positions):
  Action: 0.2522
  Action-Adventure: 0.2315
  Adventure: 0.2620
  Atmospheric: 0.2116
  Dark: 0.1879
  Exploration: 0.2284
  Female Protagonist: 0.2451
  Gore: 0.2001
  Great Soundtrack: 0.1922
  Heist: 0.1664
  Multiplayer: 0.1909
  Open World: 0.2420
  Puzzle: 0.2343
  Shooter: 0.2092
  Singleplayer: 0.2427
  Stealth: 0.2320
  Story Rich: 0.2398
  Survival: 0.2235
  Third Person: 0.2406
  Violent: 0.2130


In [4]:
# Vector for genres
genre_lists = df['genres']

# Create multi-hot encoding
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
genre_vectors = mlb.fit_transform(genre_lists)
genre_norm = normalize(genre_vectors, norm='l2')

In [5]:
# Vector for description
from sentence_transformers import SentenceTransformer

# Load multilingual model as the descriptions are in multiple languages
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
descriptions = df['short_description'].tolist()

description_vectors = model.encode(descriptions, show_progress_bar=True, normalize_embeddings=True)

print(f"Description matrix shape: {description_vectors.shape}")

/home/axel/grec/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 555.92it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 707/707 [00:20<00:00, 34.50it/s]

Description matrix shape: (22620, 384)


In [6]:
# Concatenate all the vectors

# Define weights
w_tags = 0.6
w_genres = 0.2
w_desc = 0.2

# Weight each component
combined_vectors = np.hstack([
    w_tags * tag_vectors_norm,
    w_genres * genre_norm,
    w_desc * description_vectors
])

# Final normalization
combined_vectors = normalize(combined_vectors, norm='l2')

To qualitatively evaluate the content-based similarity model, we retrieve the top-10 most similar games to ELDEN RING using cosine similarity.

In [7]:
# Testing cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
def find_similar_games(game_name, df, combined_vectors, top_n=10):
    """Find the most similar games to a given game by name."""
    
    # Find the game index by name
    matches = df[df['name'].str.lower() == game_name.lower().strip()]
    
    if matches.empty:
        print(f"No game found with name containing '{game_name}'")
        return
    
    for game_idx in matches.index:
        actual_name = df.loc[game_idx, 'name']
        app_id = df.loc[game_idx, 'app_id']
        
        print(f"🎮 Finding games similar to: {actual_name} (app_id: {app_id})")
        print("-" * 50)
        
        # Compute similarities
        similarities = cosine_similarity([combined_vectors[game_idx]], combined_vectors)[0]
        
        # Get top N most similar (excluding itself)
        top_indices = similarities.argsort()[-(top_n + 1):-1][::-1]
        
        for rank, idx in enumerate(top_indices, 1):
            print(f"{rank}. {df.loc[idx, 'name']}: {similarities[idx]:.4f}")
        
find_similar_games("ELDEN RING", df, combined_vectors)

🎮 Finding games similar to: ELDEN RING (app_id: 1245620)
--------------------------------------------------
1. DARK SOULS™ II: Scholar of the First Sin: 0.7321
2. DARK SOULS™ III: 0.7241
3. Bound By Flame: 0.6977
4. STRANGER OF PARADISE FINAL FANTASY ORIGIN: 0.6898
5. Mortal Shell: 0.6890
6. Monster Hunter: World: 0.6882
7. DARK SOULS™ II: 0.6846
8. The Last Hero of Nostalgaia: 0.6794
9. Monster Hunter Wilds: 0.6728
10. Dragon's Dogma: Dark Arisen: 0.6716


The retrieved games align very well with the core gameplay identity of Elden Ring. Tags are working as intended and the description embeddings are helping, games like Dragon's Dogma or Monster Hunter are not souls games but share things like bosses, fantasy world design, etc.

Now that there is a baseline of a working system, we can start exploring different ways to improve it.

In [8]:
# TF-IDF Approach for Tags. 
# TF-IDF improves tag representation by down-weighting tags that appear in many games
# (low discriminative power) and up-weighting rare, more informative tags.
from sklearn.feature_extraction.text import TfidfTransformer

# Apply TF-IDF transformation
tfidf_transformer = TfidfTransformer(norm='l2', use_idf=True)
tag_vectors_tfidf = tfidf_transformer.fit_transform(tag_vectors).toarray()

# Combine with genres and descriptions (using TF-IDF for tags)
combined_vectors_tfidf = np.hstack([
    w_tags * tag_vectors_tfidf,
    w_genres * genre_norm,
    w_desc * description_vectors
])
combined_vectors_tfidf = normalize(combined_vectors_tfidf, norm='l2')

find_similar_games("ELDEN RING", df, combined_vectors_tfidf)

🎮 Finding games similar to: ELDEN RING (app_id: 1245620)
--------------------------------------------------
1. DARK SOULS™ III: 0.7209
2. DARK SOULS™ II: Scholar of the First Sin: 0.7085
3. Mortal Shell: 0.6912
4. Bound By Flame: 0.6742
5. The Last Hero of Nostalgaia: 0.6668
6. STRANGER OF PARADISE FINAL FANTASY ORIGIN: 0.6545
7. CODE VEIN: 0.6484
8. Pascal's Wager: Definitive Edition: 0.6474
9. DARK SOULS™: REMASTERED: 0.6426
10. Lords Of The Fallen™ 2014: 0.6423


Using TF-IDF gives better results because the retrieved games are more clearly aligned with the Soulslike subgenre rather than just the broader action RPG category. By down-weighting very common tags such as “Action” or “RPG” and giving more importance to distinctive tags, the model shifts its focus toward games that share core mechanical and structural traits with ELDEN RING.

In [ ]:
def calculate_wilson_score(positive, negative, z=1.96):
    """
    Calculate lower bound of Wilson score confidence interval.
    Handles games with reviews better than simple positive/negative ratios.
    
    Args:
        positive: Number of positive reviews
        negative: Number of negative reviews
        z: Z-score for confidence level (1.96 = 95% confidence)
    """
    n = positive + negative
    phat = positive / n
    return (phat + z*z/(2*n) - z * np.sqrt((phat*(1-phat) + z*z/(4*n)) / n)) / (1 + z*z/n)


# Pre-calculate Wilson scores o
df['wilson_score'] = df.apply(
    lambda row: calculate_wilson_score(row['positive'], row['negative']), 
    axis=1
)

def find_similar_games(game_name, df, combined_vectors, top_n=10, quality_power=1.0):
    """
    Find similar games using hybrid scoring: similarity × quality^power.
    
    This multiplicative approach ensures:
    - Zero similarity = zero score (irrelevant games never surface)
    - Quality acts as a boost/penalty on relevant games
    
    Args:
        game_name: Name of the target game
        df: DataFrame with game data (must have 'wilson_score' pre-calculated)
        combined_vectors: Feature vectors for similarity computation
        top_n: Number of results to return
        quality_power: Exponent for quality influence (default 1.0)
    
    Returns:
        Dict mapping game indices to their top_n similar game indices,
        or None if no games found
    """
    matches = df[df['name'].str.lower() == game_name.lower().strip()]
    
    if matches.empty:
        print(f"❌ No game found with name '{game_name}'")
        return None
    
    wilson_scores = df['wilson_score'].values
    results = {}
    
    for game_idx in matches.index:
        target_game = df.loc[game_idx]
        
        print(f"\n🎮 Finding games similar to: {target_game['name']}")
        if 'app_id' in df.columns:
            print(f"   App ID: {target_game['app_id']}")
        print("-" * 80)
        
        similarities = cosine_similarity([combined_vectors[game_idx]], combined_vectors)[0]
        
        # Apply multiplicative quality boost
        hybrid_scores = similarities * (wilson_scores ** quality_power)
        
        hybrid_scores[game_idx] = -1
        top_indices = hybrid_scores.argsort()[-(top_n):][::-1]
        
        print(f"{'Rank':<5} {'Game Name':<42} {'Hybrid':<9} {'Sim':<9} {'Rating':<12} {'Reviews':<10}")
        print("=" * 90)
        
        for rank, idx in enumerate(top_indices, 1):
            name = df.loc[idx, 'name'][:40]
            h_score = hybrid_scores[idx]
            sim_score = similarities[idx]
            
            total_reviews = df.loc[idx, 'positive'] + df.loc[idx, 'negative']
            rating_pct = (df.loc[idx, 'positive'] / total_reviews * 100) if total_reviews > 0 else 0
            
            print(f"{rank:<5} {name:<42} {h_score:.4f}    {sim_score:.4f}    "
                  f"{rating_pct:>5.1f}%       {total_reviews:>,}")
        
        results[game_idx] = top_indices
    
    return results

find_similar_games("ELDEN RING", df, combined_vectors, top_n=30, quality_power=1.0)

# Pure similarity (ignore reviews)
# find_similar_games("ELDEN RING", df, combined_vectors, quality_power=0.0)

# Strong quality bias (heavily penalizes poorly-reviewed games)
# find_similar_games("ELDEN RING", df, combined_vectors, quality_power=2.0)


🎮 Finding games similar to: ELDEN RING
   App ID: 1245620
--------------------------------------------------------------------------------
Rank  Game Name                                  Hybrid    Sim       Rating       Reviews   
1     DARK SOULS™ II: Scholar of the First Sin   0.7321    0.7321     84.0%       116,710
2     DARK SOULS™ III                            0.7241    0.7241     94.3%       413,775
3     Bound By Flame                             0.6977    0.6977     68.2%       3,809
4     STRANGER OF PARADISE FINAL FANTASY ORIGI   0.6898    0.6898     84.3%       3,639
5     Mortal Shell                               0.6890    0.6890     69.3%       8,596
6     Monster Hunter: World                      0.6882    0.6882     88.0%       493,390
7     DARK SOULS™ II                             0.6846    0.6846     88.4%       44,324
8     The Last Hero of Nostalgaia                0.6794    0.6794     79.9%       1,779
9     Monster Hunter Wilds                       0.6728 

{21267: array([ 1763, 20522,  7413,  6403, 16205,  9361,  6663,  3448,   329,
        20563, 17342,  5261,   411, 21835, 15077,  3938,  2671,  6473,
         8954, 18299,  3377, 12581, 14013,  7125,  9713, 18316,  8721,
         6443,  9001, 20426])}